# 필요한 서드 파티 라이브러리 호출

- **cv2**  
  - 카메라로 이미지를 받아오고, 해당 이미지 위에 이미지 객체를 표시

- **mediapipe**  
  - Google에서 제공하는 머신러닝 기반의 얼굴 인식 라이브러리  
  - 얼굴의 랜드마크(눈, 코, 입 등 특징점)를 검출

- **numpy**  
  - 수치 계산을 위한 파이썬 라이브러리  

- **math**  
  - 삼각 함수 및 각도 계산에 사용되는 표준 수학 라이브러리  
  - 마스크 회전, 각도 측정 등에 사용

- **datetime**  
  - 현재 날짜와 시간을 가져오는 표준 라이브러리  
  - 이미지 저장 시 파일 이름을 현재 시간 기준으로 지정할 때 사용


In [1]:
import cv2
import mediapipe as mp
import numpy as np
import math
from datetime import datetime

# 본 프로젝트 메서드 정의



In [2]:
def main():
    # 카메라 연결 (0은 기본 카메라)
    cap = cv2.VideoCapture(0)
    
    # 카메라가 제대로 열렸는지 확인
    if not cap.isOpened():
        print("카메라를 열 수 없습니다.")
        return
    
    # 선글라스 이미지 로드 (투명도 지원)
    try:
        sunglasses = cv2.imread('SunGlasses.png', cv2.IMREAD_UNCHANGED)
        if sunglasses is None:
            return
    except:
        return
    
    try:
        mask = cv2.imread('mask.png', cv2.IMREAD_UNCHANGED)
        if mask is None:
            return
    except:
        return
    
    # 선글라스와 마스크 표시 상태
    show_sunglasses = False
    show_mask = False
    
    # MediaPipe Face Mesh 초기화
    mp_face_mesh = mp.solutions.face_mesh
    mp_drawing = mp.solutions.drawing_utils
    mp_drawing_styles = mp.solutions.drawing_styles
    
    # Face Mesh 모델 생성
    face_mesh = mp_face_mesh.FaceMesh(
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )
    
    # 얼굴 특징별 랜드마크 인덱스 정의
    LEFT_EYE = [362, 382, 381, 380, 374, 373, 390, 249, 263, 466, 388, 387, 386, 385, 384, 398]
    RIGHT_EYE = [33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246]
    # 입술 외곽선 랜드마크 (실제 입술 경계)
    LIPS_OUTER = [61, 146, 91, 181, 84, 17, 314, 405, 320, 307, 375, 321, 308, 324, 318]
    
    # 선글라스 위치 계산을 위한 주요 랜드마크
    LEFT_EYE_CENTER = [468]  # 왼쪽 눈 중심
    RIGHT_EYE_CENTER = [473]  # 오른쪽 눈 중심
    NOSE_BRIDGE = [6]  # 코 다리
    FOREHEAD_CENTER = [9]  # 이마 중심
    
    print("카메라가 시작되었습니다.")
    print("'1'을 눌러 선글라스 토글, '2'를 눌러 마스크 토글, 'q'를 눌러 종료하세요.")
    print("스페이스바를 눌러 스크린샷을 저장하세요.")
    
    # 트랙바가 있는 윈도우 생성
    cv2.namedWindow('MediaPipe Face Features with Sunglasses and Mask')
    
    # 입술 투명도 조절 트랙바 생성 (0-50% 범위)
    cv2.createTrackbar('Lip Opacity', 'MediaPipe Face Features with Sunglasses and Mask', 25, 50, lambda x: None)
    
    def apply_sunglasses(frame, landmarks):
        """선글라스를 얼굴에 맞춰 적용하는 함수"""
        if not show_sunglasses:
            return frame
            
        # 주요 랜드마크 좌표 추출
        left_eye_center = landmarks[468]  # 왼쪽 눈 중심
        right_eye_center = landmarks[473]  # 오른쪽 눈 중심
        left_eye_outer = landmarks[33]   # 왼쪽 눈 바깥쪽
        right_eye_outer = landmarks[362] # 오른쪽 눈 바깥쪽
        
        # 실제 눈 사이의 거리 계산
        eye_distance = np.sqrt((right_eye_center[0] - left_eye_center[0])**2 + 
                              (right_eye_center[1] - left_eye_center[1])**2)
        
        # 얼굴의 좌우 회전 각도 계산 (Y축 회전 - 좌우 돌림)
        # 코끝과 눈 중심의 상대적 위치로 좌우 회전 감지
        nose_tip = landmarks[1]
        face_center_x = (left_eye_center[0] + right_eye_center[0]) / 2
        nose_offset = nose_tip[0] - face_center_x
        
        # 정면일 때 0, 왼쪽으로 돌리면 음수, 오른쪽으로 돌리면 양수
        y_rotation_factor = nose_offset / eye_distance  # -1 ~ 1 범위로 정규화
        y_rotation_factor = max(-0.8, min(0.8, y_rotation_factor))  # 범위 제한
        
        # 선글라스 크기 조정 (눈 사이 거리에 비례, 더 정확한 비율)
        sunglasses_width = int(eye_distance * 2.4)  # 안경 전체 너비
        sunglasses_height = int(sunglasses_width * sunglasses.shape[0] / sunglasses.shape[1])
        
        # 3D 원근법 적용을 위한 변형
        # 얼굴이 돌아간 정도에 따라 안경의 좌우 크기를 다르게 조정
        if abs(y_rotation_factor) > 0.1:  # 회전이 감지된 경우
            # 원근법 적용: 먼 쪽은 작게, 가까운 쪽은 크게
            perspective_strength = abs(y_rotation_factor) * 0.3  # 원근법 강도
            
            # 4개의 모서리 점 정의
            h, w = sunglasses.shape[:2]
            src_points = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
            
            if y_rotation_factor > 0:  # 오른쪽으로 돌린 경우 (왼쪽이 멀어짐)
                # 왼쪽을 작게, 오른쪽을 크게
                dst_points = np.float32([
                    [perspective_strength * w * 0.2, perspective_strength * h * 0.1],  # 왼쪽 위
                    [w, 0],  # 오른쪽 위
                    [w, h],  # 오른쪽 아래
                    [perspective_strength * w * 0.2, h - perspective_strength * h * 0.1]  # 왼쪽 아래
                ])
            else:  # 왼쪽으로 돌린 경우 (오른쪽이 멀어짐)
                # 오른쪽을 작게, 왼쪽을 크게
                dst_points = np.float32([
                    [0, 0],  # 왼쪽 위
                    [w - perspective_strength * w * 0.2, perspective_strength * h * 0.1],  # 오른쪽 위
                    [w - perspective_strength * w * 0.2, h - perspective_strength * h * 0.1],  # 오른쪽 아래
                    [0, h]   # 왼쪽 아래
                ])
            
            # 원근법 변환 적용
            perspective_matrix = cv2.getPerspectiveTransform(src_points, dst_points)
            resized_sunglasses = cv2.resize(sunglasses, (sunglasses_width, sunglasses_height))
            warped_sunglasses = cv2.warpPerspective(resized_sunglasses, perspective_matrix, 
                                                  (sunglasses_width, sunglasses_height))
        else:
            # 정면에 가까울 때는 일반 리사이즈
            warped_sunglasses = cv2.resize(sunglasses, (sunglasses_width, sunglasses_height))
        
        # 두 눈의 중점 계산 (중심점 기준)
        eye_center_x = int((left_eye_center[0] + right_eye_center[0]) / 2)
        eye_center_y = int((left_eye_center[1] + right_eye_center[1]) / 2 + 10)
        
        # 얼굴의 Z축 회전 각도 계산 (기울임)
        angle = math.atan2(right_eye_center[1] - left_eye_center[1], 
                          right_eye_center[0] - left_eye_center[0])
        angle_degrees = -math.degrees(angle)  # 회전 방향 반전
        
        # Z축 회전 적용 (기울임)
        h, w = warped_sunglasses.shape[:2]
        center = (w // 2, h // 2)
        rotation_matrix = cv2.getRotationMatrix2D(center, angle_degrees, 1.0)
        
        # 회전된 이미지의 새로운 경계 계산
        cos_val = abs(rotation_matrix[0, 0])
        sin_val = abs(rotation_matrix[0, 1])
        new_w = int((h * sin_val) + (w * cos_val))
        new_h = int((h * cos_val) + (w * sin_val))
        
        # 회전 중심 조정
        rotation_matrix[0, 2] += (new_w / 2) - center[0]
        rotation_matrix[1, 2] += (new_h / 2) - center[1]
        
        # 최종 회전 적용
        rotated_sunglasses = cv2.warpAffine(warped_sunglasses, rotation_matrix, (new_w, new_h))
        
        # 선글라스를 배치할 위치 계산
        y_offset = int(sunglasses_height * 0.05)  # 약간만 위로
        start_x = eye_center_x - new_w // 2
        start_y = eye_center_y - new_h // 2 - y_offset
        
        # 프레임 경계 확인 및 조정
        if start_x < 0:
            rotated_sunglasses = rotated_sunglasses[:, -start_x:]
            start_x = 0
        if start_y < 0:
            rotated_sunglasses = rotated_sunglasses[-start_y:, :]
            start_y = 0
        if start_x + rotated_sunglasses.shape[1] > frame.shape[1]:
            rotated_sunglasses = rotated_sunglasses[:, :frame.shape[1] - start_x]
        if start_y + rotated_sunglasses.shape[0] > frame.shape[0]:
            rotated_sunglasses = rotated_sunglasses[:frame.shape[0] - start_y, :]
        
        # 선글라스 적용
        if rotated_sunglasses.shape[2] == 4:  # RGBA
            # 알파 채널이 있는 경우
            for c in range(0, 3):
                alpha = rotated_sunglasses[:, :, 3] / 255.0
                frame[start_y:start_y + rotated_sunglasses.shape[0], 
                      start_x:start_x + rotated_sunglasses.shape[1], c] = \
                    (alpha * rotated_sunglasses[:, :, c] + 
                     (1 - alpha) * frame[start_y:start_y + rotated_sunglasses.shape[0], 
                                        start_x:start_x + rotated_sunglasses.shape[1], c])
        else:  # RGB
            # 알파 채널이 없는 경우
            mask_alpha = np.all(rotated_sunglasses[:, :, :3] > [10, 10, 10], axis=2)
            frame[start_y:start_y + rotated_sunglasses.shape[0], 
                  start_x:start_x + rotated_sunglasses.shape[1]][mask_alpha] = \
                rotated_sunglasses[:, :, :3][mask_alpha]
        
        return frame
    
    def apply_mask(frame, landmarks):
        """마스크를 코 부분에 맞춰 적용하는 함수"""
        if not show_mask:
            return frame
        
        # 코 관련 주요 랜드마크
        nose_bridge = landmarks[6]   # 코 다리
        nose_tip = landmarks[1]      # 코끝
        left_nostril = landmarks[235]  # 왼쪽 콧구멍
        right_nostril = landmarks[455]  # 오른쪽 콧구멍
        
        # 눈 중심점
        left_eye_center = landmarks[468]
        right_eye_center = landmarks[473]
        
        # 코의 너비와 높이 계산
        nose_width = np.sqrt((right_nostril[0] - left_nostril[0])**2 + 
                            (right_nostril[1] - left_nostril[1])**2)
        nose_height = np.sqrt((nose_tip[0] - nose_bridge[0])**2 + 
                             (nose_tip[1] - nose_bridge[1])**2)
        
        # 마스크 크기 조정
        mask_width = int(nose_width * 4)  # 마스크 너비
        mask_height = int(mask_width * mask.shape[0] / mask.shape[1])
        
        # 마스크 리사이즈
        resized_mask = cv2.resize(mask, (mask_width, mask_height))
        
        # 마스크 중심점 계산 (코 중간 부분)
        mask_center_x = int((left_nostril[0] + right_nostril[0]) / 2)
        mask_center_y = int((nose_bridge[1] + nose_tip[1]) / 2)
        
        # 얼굴의 Z축 회전 각도 계산 (기울임)
        angle = math.atan2(right_eye_center[1] - left_eye_center[1], 
                          right_eye_center[0] - left_eye_center[0])
        angle_degrees = -math.degrees(angle)
        
        # 마스크 회전 적용
        h, w = resized_mask.shape[:2]
        center = (w // 2, h // 2)
        rotation_matrix = cv2.getRotationMatrix2D(center, angle_degrees, 1.0)
        
        # 회전된 이미지의 새로운 경계 계산
        cos_val = abs(rotation_matrix[0, 0])
        sin_val = abs(rotation_matrix[0, 1])
        new_w = int((h * sin_val) + (w * cos_val))
        new_h = int((h * cos_val) + (w * sin_val))
        
        # 회전 중심 조정
        rotation_matrix[0, 2] += (new_w / 2) - center[0]
        rotation_matrix[1, 2] += (new_h / 2) - center[1]
        
        # 최종 회전 적용
        rotated_mask = cv2.warpAffine(resized_mask, rotation_matrix, (new_w, new_h))
        
        # 마스크를 배치할 위치 계산
        start_x = mask_center_x - new_w // 2 + 2
        start_y = mask_center_y - new_h // 2 + 15
        
        # 프레임 경계 확인 및 조정
        if start_x < 0:
            rotated_mask = rotated_mask[:, -start_x:]
            start_x = 0
        if start_y < 0:
            rotated_mask = rotated_mask[-start_y:, :]
            start_y = 0
        if start_x + rotated_mask.shape[1] > frame.shape[1]:
            rotated_mask = rotated_mask[:, :frame.shape[1] - start_x]
        if start_y + rotated_mask.shape[0] > frame.shape[0]:
            rotated_mask = rotated_mask[:frame.shape[0] - start_y, :]
        
        for c in range(0, 3):
                alpha = rotated_mask[:, :, 3] / 255.0
                frame[start_y:start_y + rotated_mask.shape[0], 
                      start_x:start_x + rotated_mask.shape[1], c] = \
                    (alpha * rotated_mask[:, :, c] + 
                     (1 - alpha) * frame[start_y:start_y + rotated_mask.shape[0], 
                                        start_x:start_x + rotated_mask.shape[1], c])
        
        return frame
    
    # 립스틱 적용 함수
    def apply_lips(frame, landmarks):
        # 트랙바에서 투명도 값 가져오기 (0-50)
        opacity = cv2.getTrackbarPos('Lip Opacity', 'MediaPipe Face Features with Sunglasses and Mask')
        
        if opacity == 0:
            return frame
        
        # 투명도를 0-1 범위로 변환
        alpha = opacity / 100.0  # 0-50을 0-0.5로 변환
        
        # 입술 외곽선 랜드마크 포인트들 (실제 입술 경계)
        lip_points = np.array([landmarks[i] for i in LIPS_OUTER], np.int32)
        
        # 오버레이를 위한 임시 이미지 생성
        overlay = frame.copy()
        
        # 입술 영역을 빨간색으로 채우기
        cv2.fillPoly(overlay, [lip_points], (0, 0, 255))  # BGR 순서로 빨간색
        
        # 원본 이미지와 오버레이를 블렌딩
        cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)
        
        return frame
    
    while True:
        # 프레임 읽기
        ret, frame = cap.read()

        if not ret:
            print("프레임 Error.")
            break
        
        # 프레임을 좌우 반전
        frame = cv2.flip(frame, 1)
        
        # MediaPipe 처리를 위해 RGB로 변환
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # 얼굴 랜드마크 검출
        results = face_mesh.process(rgb_frame)
        
        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                # 랜드마크 좌표를 픽셀 좌표로 변환
                h, w, _ = frame.shape
                landmarks = []
                for landmark in face_landmarks.landmark:
                    x = int(landmark.x * w)
                    y = int(landmark.y * h)
                    landmarks.append((x, y))
                
                # 선글라스 적용
                frame = apply_sunglasses(frame, landmarks)
                
                # 마스크 적용
                frame = apply_mask(frame, landmarks)
                
                # 입술 적용 (항상 적용, 트랙바로 투명도 조절)
                frame = apply_lips(frame, landmarks)
                
                # 얼굴 특징 그리기 (확인용)
                if not show_sunglasses:
                    # 왼쪽 눈 그리기
                    eye_points = np.array([landmarks[i] for i in LEFT_EYE], np.int32)
                    cv2.fillPoly(frame, [eye_points], (0, 0, 255))
                    cv2.putText(frame, 'L-Eye', (landmarks[LEFT_EYE[0]][0]-20, landmarks[LEFT_EYE[0]][1]-10), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
                    
                    # 오른쪽 눈 그리기
                    eye_points = np.array([landmarks[i] for i in RIGHT_EYE], np.int32)
                    cv2.fillPoly(frame, [eye_points], (0, 0, 255))
                    cv2.putText(frame, 'R-Eye', (landmarks[RIGHT_EYE[0]][0]-20, landmarks[RIGHT_EYE[0]][1]-10), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
                
                if not show_mask:
                    # 코 표시 (코끝)
                    nose_tip = landmarks[1]  # 코끝 랜드마크
                    cv2.circle(frame, nose_tip, 8, (0, 0, 255), -1)
                    cv2.putText(frame, 'Nose', (nose_tip[0]-20, nose_tip[1]-15), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        
        # 키 입력 처리
        key = cv2.waitKey(1) & 0xFF
        
        if key == ord(' '):  # 스페이스바 - 스크린샷 저장 (텍스트 없이)
            # 깨끗한 프레임 복사본 만들기
            clean_frame = frame.copy()
            
            # 타임스탬프로 파일명 생성
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"capture_{timestamp}.jpg"
            cv2.imwrite(filename, clean_frame)
            print(f"스크린샷 저장됨: {filename}")
            
        elif key == ord('q'):
            break
        elif key == ord('1'):
            show_sunglasses = not show_sunglasses
            print(f"선글라스 {'ON' if show_sunglasses else 'OFF'}")
        elif key == ord('2'):
            show_mask = not show_mask
            print(f"마스크 {'ON' if show_mask else 'OFF'}")
        
        # 화면에 표시할 프레임에 텍스트 추가
        display_frame = frame.copy()
        
        # 상태 정보 표시
        sunglasses_status = "Sunglasses: ON" if show_sunglasses else "Sunglasses: OFF"
        mask_status = "Mask: ON" if show_mask else "Mask: OFF"
        lip_opacity = cv2.getTrackbarPos('Lip Opacity', 'MediaPipe Face Features with Sunglasses and Mask')
        
        cv2.putText(display_frame, sunglasses_status, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(display_frame, mask_status, (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(display_frame, f"Lip Opacity: {lip_opacity}%", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(display_frame, "Press '1' for sunglasses, '2' for mask, SPACE for screenshot, 'q' to quit", (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        # 화면에 표시 (텍스트가 포함된 프레임)
        cv2.imshow('MediaPipe Face Features with Sunglasses and Mask', display_frame)
    
    # 자원 해제
    cap.release()
    cv2.destroyAllWindows()
    print("카메라가 종료되었습니다.")

# main 실행

In [3]:
if __name__ == "__main__":
    main()

카메라가 시작되었습니다.
'1'을 눌러 선글라스 토글, '2'를 눌러 마스크 토글, 'q'를 눌러 종료하세요.
스페이스바를 눌러 스크린샷을 저장하세요.
선글라스 ON
마스크 ON
선글라스 OFF
마스크 OFF
선글라스 ON
마스크 ON
카메라가 종료되었습니다.
